# HMM validation (grasp vs. non-grasp)

Dieses Notebook trainiert 10 HMM-Modelle (n_states=3..12) auf den Trainings-CSV-Dateien, führt pro Modell auto_grasp_state aus und validiert anschließend gegen val_csv mit der Ground-Truth-Spalte aktion (0/1).

Harte Qualitätsgrenze: precision_grasp_micro >= 0.90.

Begründung für diese Grenze: Greif-Vorhersagen sollen verlässlich sein und zu viele False Positives vermeiden.

Zweites Ranking-Kriterium unter den zulässigen Modellen: recall_grasp_micro.

Interpretation: recall_grasp = TP / (TP + FN) beantwortet die Frage: „Wie viele echte Greif-Frames werden erkannt?“

Tie-Breaker: f1_grasp_micro, balanced_accuracy_micro, mcc_micro.

Reporting-Strategie: Alle Metriken bleiben sichtbar, um sowohl False Positives als auch verpasste Greif-Frames zu überwachen.


In [1]:
from __future__ import annotations

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

from handpose.config_loader import ROOT_DIR, get_settings
from handpose.backend.ML.HMM.features_hmm import HmmFeatureExtractor
from handpose.backend.ML.HMM.model_hmm import HmmWrapper
from handpose.backend.ML.DBSCAN.auto_grasp_state import detect_grasp_state_by_mean_max_reach_y

/Users/jole/Library/Caches/pypoetry/virtualenvs/handpose-estimation-ZvAsxI9K-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
training_dir_preferred = ROOT_DIR / "data/03_processed/features/training"
training_dir_fallback = ROOT_DIR / "data/03_processed/keypoints/training"
validation_dir = ROOT_DIR / "data/05_validation/val_csv"

settings = get_settings()
HMM_N_ITER = settings.hmm.n_iter
HMM_RANDOM_STATE = settings.hmm.random_state
FEATURE_INDEX_MAX_REACH_Y = settings.dbscan.feature_index_max_reach_y
MIN_PRECISION_GRASP_MICRO = 0.90
SECONDARY_VALIDATION_METRIC = "recall_grasp_micro"
PRIMARY_VALIDATION_METRIC = SECONDARY_VALIDATION_METRIC
RANKING_COLUMNS = [
    "validation_score",
    "f1_grasp_micro",
    "balanced_accuracy_micro",
    "mcc_micro",
]
RANKING_ASCENDING = [False] * len(RANKING_COLUMNS)

STATE_RANGE = range(3, 13)

print(f'ROOT_DIR: {ROOT_DIR}')
print(f'Preferred training dir: {training_dir_preferred}')
print(f'Fallback training dir: {training_dir_fallback}')
print(f'Validation dir: {validation_dir}')

ROOT_DIR: /Users/jole/Studium/HandPose/HandPoseEstimation
Preferred training dir: /Users/jole/Studium/HandPose/HandPoseEstimation/data/03_processed/features/training
Fallback training dir: /Users/jole/Studium/HandPose/HandPoseEstimation/data/03_processed/keypoints/training
Validation dir: /Users/jole/Studium/HandPose/HandPoseEstimation/data/05_validation/val_csv


In [3]:
REQUIRED_KEYPOINT_COLS = {"l_x_8", "l_y_8", "r_x_8", "r_y_8", "l_x_4", "l_y_4", "r_x_4", "r_y_4"}


def has_required_keypoint_columns(csv_path: Path) -> bool:
    sample = pd.read_csv(csv_path, sep=";", nrows=1)
    return REQUIRED_KEYPOINT_COLS.issubset(set(sample.columns))


def resolve_training_dir(preferred_dir: Path, fallback_dir: Path) -> Path:
    preferred_files = sorted(preferred_dir.glob("*.csv"))
    if preferred_files and has_required_keypoint_columns(preferred_files[0]):
        return preferred_dir

    fallback_files = sorted(fallback_dir.glob("*.csv"))
    if fallback_files and has_required_keypoint_columns(fallback_files[0]):
        print(
            "INFO: Preferred training dir has no raw keypoint columns required by HmmFeatureExtractor."
        )
        print(f"Using fallback dir for HMM training: {fallback_dir}")
        return fallback_dir

    raise ValueError(
        "No usable training directory found. Neither preferred nor fallback has required keypoint columns."
    )


def extract_action_labels(df: pd.DataFrame) -> np.ndarray:
    for col in ["aktion", "Aktion", "action", "Action"]:
        if col in df.columns:
            y = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int).to_numpy()
            return (y > 0).astype(int)
    raise KeyError("No action column found. Expected one of: aktion, Aktion, action, Action")


def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    unique_true = np.unique(y_true)
    unique_pred = np.unique(y_pred)

    if len(unique_true) > 1:
        bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    else:
        bal_acc = float("nan")

    if len(unique_true) > 1 and len(unique_pred) > 1:
        mcc = float(matthews_corrcoef(y_true, y_pred))
    else:
        mcc = 0.0

    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": bal_acc,
        "precision_grasp": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall_grasp": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_grasp": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "specificity_non_grasp": float(specificity),
        "mcc": mcc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [4]:
# Load training data and precompute HMM feature sequences
extractor = HmmFeatureExtractor()
training_dir = resolve_training_dir(training_dir_preferred, training_dir_fallback)
training_files = sorted(training_dir.glob("*.csv"))
validation_files = sorted(validation_dir.glob("*.csv"))

if not training_files:
    raise FileNotFoundError(f"No training CSV files found in {training_dir}")
if not validation_files:
    raise FileNotFoundError(f"No validation CSV files found in {validation_dir}")

X_train_parts: list[np.ndarray] = []
train_lengths: list[int] = []

for csv_path in training_files:
    df = extractor.process_csv(csv_path)
    if df is None:
        continue

    X_seq, _ = extractor.calculate_features(df)
    if len(X_seq) == 0:
        continue

    X_train_parts.append(X_seq)
    train_lengths.append(len(X_seq))

if not X_train_parts:
    raise RuntimeError("Could not extract any training features.")

X_train_concat = np.concatenate(X_train_parts, axis=0)

validation_data = []
for csv_path in validation_files:
    df_val = extractor.process_csv(csv_path)
    if df_val is None:
        continue

    X_val, _ = extractor.calculate_features(df_val)
    y_true = extract_action_labels(df_val)

    seq_len = min(len(X_val), len(y_true))
    if seq_len == 0:
        continue

    validation_data.append({
        "file_name": csv_path.name,
        "X": X_val[:seq_len],
        "y_true": y_true[:seq_len],
    })

if not validation_data:
    raise RuntimeError("No usable validation data loaded.")

print(f"Training directory used: {training_dir}")
print(f"Training files loaded: {len(train_lengths)}")
print(f"Validation files loaded: {len(validation_data)}")
print(f"Training frames total: {len(X_train_concat)}")

INFO: Preferred training dir has no raw keypoint columns required by HmmFeatureExtractor.
Using fallback dir for HMM training: /Users/jole/Studium/HandPose/HandPoseEstimation/data/03_processed/keypoints/training
Training directory used: /Users/jole/Studium/HandPose/HandPoseEstimation/data/03_processed/keypoints/training
Training files loaded: 501
Validation files loaded: 7
Training frames total: 119658


/Users/jole/Studium/HandPose/HandPoseEstimation/src/handpose/backend/ML/HMM/features_hmm.py:33: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df.interpolate(method="linear", limit_direction="both")
/Users/jole/Studium/HandPose/HandPoseEstimation/src/handpose/backend/ML/HMM/features_hmm.py:33: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df.interpolate(method="linear", limit_direction="both")
/Users/jole/Studium/HandPose/HandPoseEstimation/src/handpose/backend/ML/HMM/features_hmm.py:33: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df.interpolate(method="linear", limit_direction="both")
/Users/jole/Studium/HandPose

In [5]:
# Train and evaluate models for n_states = 3..12
per_file_rows: list[dict] = []
model_rows: list[dict] = []

for n_states in STATE_RANGE:
    print(f"\n=== Training HMM with n_states={n_states} ===")

    wrapper = HmmWrapper(
        n_states=n_states,
        n_iter=HMM_N_ITER,
        random_state=HMM_RANDOM_STATE,
    )
    wrapper.train(X_train_concat, train_lengths)

    with tempfile.TemporaryDirectory(prefix=f"hmm_states_{n_states}_") as tmp_dir_str:
        tmp_dir = Path(tmp_dir_str)
        model_path = tmp_dir / "hmm_model.pkl"
        scaler_path = tmp_dir / "scaler.pkl"
        wrapper.save(str(model_path), str(scaler_path))

        grasp_state, state_means, files_used_for_grasp = detect_grasp_state_by_mean_max_reach_y(
            raw_data_dir=training_dir,
            hmm_model_dir=tmp_dir,
            feature_index_max_reach_y=FEATURE_INDEX_MAX_REACH_Y,
        )

    if grasp_state is None:
        print(f"WARN: Could not detect grasp_state for n_states={n_states}. Skipping evaluation.")
        continue

    y_true_all: list[np.ndarray] = []
    y_pred_all: list[np.ndarray] = []

    for item in validation_data:
        states = wrapper.predict(item["X"])
        y_pred = (states == grasp_state).astype(int)
        y_true = item["y_true"]

        seq_len = min(len(y_true), len(y_pred))
        y_true_eval = y_true[:seq_len]
        y_pred_eval = y_pred[:seq_len]

        metrics = compute_binary_metrics(y_true_eval, y_pred_eval)
        metrics.update({
            "n_states": n_states,
            "grasp_state": int(grasp_state),
            "file_name": item["file_name"],
            "n_frames": int(seq_len),
            "files_used_for_grasp_detection": int(files_used_for_grasp),
            "state_means": str(state_means),
        })
        per_file_rows.append(metrics)

        y_true_all.append(y_true_eval)
        y_pred_all.append(y_pred_eval)

    y_true_all_concat = np.concatenate(y_true_all, axis=0)
    y_pred_all_concat = np.concatenate(y_pred_all, axis=0)

    micro = compute_binary_metrics(y_true_all_concat, y_pred_all_concat)

    per_model_df = pd.DataFrame([r for r in per_file_rows if r["n_states"] == n_states])
    macro_f1 = float(per_model_df["f1_grasp"].mean())
    macro_bal_acc = float(per_model_df["balanced_accuracy"].mean())

    model_rows.append({
        "n_states": n_states,
        "grasp_state": int(grasp_state),
        "files_used_for_grasp_detection": int(files_used_for_grasp),
        "n_frames_total": int(len(y_true_all_concat)),
        "accuracy_micro": micro["accuracy"],
        "balanced_accuracy_micro": micro["balanced_accuracy"],
        "precision_grasp_micro": micro["precision_grasp"],
        "recall_grasp_micro": micro["recall_grasp"],
        "f1_grasp_micro": micro["f1_grasp"],
        "specificity_non_grasp_micro": micro["specificity_non_grasp"],
        "mcc_micro": micro["mcc"],
        "tn_micro": micro["tn"],
        "fp_micro": micro["fp"],
        "fn_micro": micro["fn"],
        "tp_micro": micro["tp"],
        "f1_grasp_macro": macro_f1,
        "balanced_accuracy_macro": macro_bal_acc,
        # Primary validation score: maximize grasp recall (class 1)
        "validation_score": micro["recall_grasp"],
        "validation_score_name": PRIMARY_VALIDATION_METRIC,
    })


=== Training HMM with n_states=3 ===


         1 -676820.82275636             +nan
         2 -457562.66301470 +219258.15974166
         3 -445284.55224460  +12278.11077010
         4 -443924.24689086   +1360.30535374
         5 -443693.02224269    +231.22464817
         6 -443647.21414095     +45.80810174
         7 -443637.51665651      +9.69748445
         8 -443635.09757567      +2.41908083
         9 -443634.19744030      +0.90013537
        10 -443633.63335927      +0.56408103
        11 -443633.14355059      +0.48980868
        12 -443632.66089277      +0.48265782
        13 -443632.16203612      +0.49885664
        14 -443631.63404080      +0.52799532
        15 -443631.06614189      +0.56789891
        16 -443630.44725534      +0.61888655
        17 -443629.76497829      +0.68227705
        18 -443629.00509764      +0.75988064
        19 -443628.15133031      +0.85376734
        20 -443627.18510962      +0.96622069
        21 -443626.08512952      +1.09998010
        22 -443624.82622281      +1.25890671
        23

2026-03-08 16:11:13 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state | Automatische Grasp-State-Erkennung (mean_max_reach_y):
2026-03-08 16:11:13 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=0 -> mean_max_reach_y=0.447935
2026-03-08 16:11:13 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=1 -> mean_max_reach_y=0.502448
2026-03-08 16:11:13 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=2 -> mean_max_reach_y=0.729060
2026-03-08 16:11:13 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state | Erkannter Grasp-State: 2

=== Training HMM with n_states=4 ===


         1 -655926.98581811             +nan
         2 -480581.69194049 +175345.29387762
         3 -424943.57038984  +55638.12155065
         4 -410636.71007179  +14306.86031805
         5 -408333.86751751   +2302.84255428
         6 -407622.70316428    +711.16435323
         7 -407275.86038724    +346.84277704
         8 -407060.43702354    +215.42336370
         9 -406912.68135716    +147.75566637
        10 -406813.98555782     +98.69579934
        11 -406746.54554575     +67.44001208
        12 -406699.14432120     +47.40122454
        13 -406662.64802591     +36.49629530
        14 -406633.27037746     +29.37764844
        15 -406608.78090566     +24.48947180
        16 -406585.97065519     +22.81025047
        17 -406562.90650274     +23.06415245
        18 -406537.14898331     +25.75751943
        19 -406500.94204910     +36.20693421
        20 -406475.11518411     +25.82686498
        21 -406451.71052718     +23.40465694
        22 -406428.60405792     +23.10646926
        23

2026-03-08 16:11:29 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state | Automatische Grasp-State-Erkennung (mean_max_reach_y):
2026-03-08 16:11:29 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=0 -> mean_max_reach_y=0.440155
2026-03-08 16:11:29 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=1 -> mean_max_reach_y=0.721360
2026-03-08 16:11:29 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=2 -> mean_max_reach_y=0.586683
2026-03-08 16:11:29 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=3 -> mean_max_reach_y=0.486031
2026-03-08 16:11:29 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state | Erkannter Grasp-State: 1

=== Training HMM with n_states=5 ===


...

2026-03-08 16:15:57 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state | Automatische Grasp-State-Erkennung (mean_max_reach_y):
2026-03-08 16:15:57 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=0 -> mean_max_reach_y=0.631948
2026-03-08 16:15:57 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=1 -> mean_max_reach_y=0.404875
2026-03-08 16:15:57 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=2 -> mean_max_reach_y=0.453349
2026-03-08 16:15:57 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=3 -> mean_max_reach_y=0.606866
2026-03-08 16:15:57 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=4 -> mean_max_reach_y=0.479632
2026-03-08 16:15:57 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=5 -> mean_max_reach_y=0.779987
2026-03-08 16:15:57 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=6 -> mean_max_reach_y=0.512708
2026-03-08 16:15:57 | INFO     | handpos

         1 -735955.63469359             +nan
         2 -459798.79884501 +276156.83584858
         3 -395824.31655451  +63974.48229050
         4 -382284.80174535  +13539.51480916
         5 -374936.04712030   +7348.75462505
         6 -367223.63090759   +7712.41621271
         7 -360785.28230820   +6438.34859940
         8 -355930.49205310   +4854.79025510
         9 -351495.95338023   +4434.53867287
        10 -342873.88267420   +8622.07070603
        11 -332183.05687646  +10690.82579773
        12 -326665.36596283   +5517.69091363
        13 -323673.09544883   +2992.27051399
        14 -320832.41094638   +2840.68450245
        15 -316841.23525318   +3991.17569320
        16 -311013.43293337   +5827.80231981
        17 -305163.62738087   +5849.80555250
        18 -301330.89759089   +3832.72978998
        19 -298110.46476025   +3220.43283063
        20 -295466.49601086   +2643.96874940
        21 -293760.72938566   +1705.76662519
        22 -292681.10254364   +1079.62684202
        23

2026-03-08 16:17:14 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state | Automatische Grasp-State-Erkennung (mean_max_reach_y):
2026-03-08 16:17:14 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=0 -> mean_max_reach_y=0.743218
2026-03-08 16:17:14 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=1 -> mean_max_reach_y=0.565740
2026-03-08 16:17:14 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=2 -> mean_max_reach_y=0.472132
2026-03-08 16:17:14 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=3 -> mean_max_reach_y=0.767530
2026-03-08 16:17:14 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=4 -> mean_max_reach_y=0.756545
2026-03-08 16:17:14 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=5 -> mean_max_reach_y=0.465670
2026-03-08 16:17:14 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=6 -> mean_max_reach_y=0.578286
2026-03-08 16:17:14 | INFO     | handpos

         1 -712900.75327402             +nan
         2 -448076.26528851 +264824.48798550
         3 -387133.21736073  +60943.04792778
         4 -368844.39112715  +18288.82623358
         5 -358252.36488095  +10592.02624621
         6 -347737.54677964  +10514.81810131
         7 -338256.70995073   +9480.83682891
         8 -330659.77077525   +7596.93917548
         9 -323906.98568712   +6752.78508813
        10 -316349.37065478   +7557.61503234
        11 -308622.23241873   +7727.13823605
        12 -303829.37214812   +4792.86027061
        13 -300879.81101941   +2949.56112871
        14 -298755.59117161   +2124.21984780
        15 -297122.12015811   +1633.47101350
        16 -295710.50858383   +1411.61157428
        17 -294439.04806266   +1271.46052117
        18 -292948.48118885   +1490.56687381
        19 -289553.10052374   +3395.38066511
        20 -284338.11288879   +5214.98763495
        21 -280640.84718860   +3697.26570019
        22 -277623.89036810   +3016.95682051
        23

2026-03-08 16:17:51 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state | Automatische Grasp-State-Erkennung (mean_max_reach_y):
2026-03-08 16:17:51 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=0 -> mean_max_reach_y=0.392930
2026-03-08 16:17:51 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=1 -> mean_max_reach_y=0.758511
2026-03-08 16:17:51 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=2 -> mean_max_reach_y=0.601145
2026-03-08 16:17:51 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=3 -> mean_max_reach_y=0.581819
2026-03-08 16:17:51 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=4 -> mean_max_reach_y=0.469864
2026-03-08 16:17:51 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=5 -> mean_max_reach_y=0.307915
2026-03-08 16:17:51 | INFO     | handpose.backend.ML.DBSCAN.auto_grasp_state |   state=6 -> mean_max_reach_y=0.760916
2026-03-08 16:17:51 | INFO     | handpos

In [6]:
per_file_results = pd.DataFrame(per_file_rows)
model_summary_all = pd.DataFrame(model_rows)

eligible_models = model_summary_all[
    model_summary_all["precision_grasp_micro"] >= MIN_PRECISION_GRASP_MICRO
].copy()

if eligible_models.empty:
    print(
        f"WARN: No model passes precision gate (precision_grasp_micro >= {MIN_PRECISION_GRASP_MICRO:.2f}). "
        "Falling back to all models for ranking."
    )
    ranking_base = model_summary_all
else:
    print(
        f"Models passing precision gate (precision_grasp_micro >= {MIN_PRECISION_GRASP_MICRO:.2f}): "
        f"{len(eligible_models)}/{len(model_summary_all)}"
    )
    ranking_base = eligible_models

model_summary = ranking_base.sort_values(
    by=RANKING_COLUMNS,
    ascending=RANKING_ASCENDING,
).reset_index(drop=True)

if model_summary.empty:
    raise RuntimeError("No model result available.")

best_model = model_summary.iloc[0]

print(
    f"Best model with precision gate (precision_grasp_micro >= {MIN_PRECISION_GRASP_MICRO:.2f}) "
    f"by {PRIMARY_VALIDATION_METRIC}:"
)
print(best_model[["n_states", "grasp_state", "precision_grasp_micro", "validation_score", "f1_grasp_micro", "balanced_accuracy_micro", "mcc_micro"]])

display(model_summary)
display(per_file_results.sort_values(["n_states", "file_name"]).reset_index(drop=True))

Models passing precision gate (precision_grasp_micro >= 0.90): 7/10
Best model with precision gate (precision_grasp_micro >= 0.90) by recall_grasp_micro:
n_states                          8
grasp_state                       1
precision_grasp_micro      0.967914
validation_score           0.874396
f1_grasp_micro             0.918782
balanced_accuracy_micro    0.930887
mcc_micro                  0.888194
Name: 0, dtype: object


,n_states,grasp_state,files_used_for_grasp_detection,n_frames_total,accuracy_micro,balanced_accuracy_micro,precision_grasp_micro,recall_grasp_micro,f1_grasp_micro,specificity_non_grasp_micro,mcc_micro,tn_micro,fp_micro,fn_micro,tp_micro,f1_grasp_macro,balanced_accuracy_macro,validation_score,validation_score_name
0,8,1,501,2047,0.953102,0.930887,0.967914,0.874396,0.918782,0.987377,0.888194,1408,18,78,543,0.917393,0.930773,0.874396,recall_grasp_micro
1,6,2,501,2047,0.936492,0.918511,0.913997,0.872786,0.892916,0.964236,0.848267,1375,51,79,542,0.891740,0.918986,0.872786,recall_grasp_micro
2,7,2,501,2047,0.947728,0.917030,0.986742,0.838969,0.906876,0.995091,0.876410,1419,7,100,521,0.903783,0.916176,0.838969,recall_grasp_micro
3,11,3,501,2047,0.906204,0.851319,0.971429,0.711755,0.821561,0.990884,0.776892,1413,13,179,442,0.818294,0.849453,0.711755,recall_grasp_micro
4,12,10,501,2047,0.911089,0.854371,0.995485,0.710145,0.828947,0.998597,0.791207,1424,2,180,441,0.821596,0.852653,0.710145,recall_grasp_micro
5,10,5,501,2047,0.903762,0.842294,0.995327,0.685990,0.812202,0.998597,0.773908,1424,2,195,426,0.807024,0.841426,0.685990,recall_grasp_micro
6,9,8,501,2047,0.897899,0.832178,0.997585,0.665056,0.798068,0.999299,0.760348,1425,1,208,413,0.791202,0.829806,0.665056,recall_grasp_micro


,accuracy,balanced_accuracy,precision_grasp,recall_grasp,f1_grasp,specificity_non_grasp,mcc,tn,fp,fn,tp,n_states,grasp_state,file_name,n_frames,files_used_for_grasp_detection,state_means
0,0.970149,0.976047,0.910000,0.989130,0.947917,0.962963,0.928630,234,9,1,91,3,2,recording_20260225_143126.csv,335,501,"{0: 0.4479353650478908, 1: 0.5024482701784585,..."
1,0.961538,0.972973,0.882353,1.000000,0.937500,0.945946,0.913596,210,12,0,90,3,2,recording_20260225_143150.csv,312,501,"{0: 0.4479353650478908, 1: 0.5024482701784585,..."
2,0.924765,0.937440,0.809091,0.967391,0.881188,0.907489,0.833846,206,21,3,89,3,2,recording_20260225_143209.csv,319,501,"{0: 0.4479353650478908, 1: 0.5024482701784585,..."
3,0.870861,0.910138,0.685484,1.000000,0.813397,0.820276,0.749858,178,39,0,85,3,2,recording_20260225_143246.csv,302,501,"{0: 0.4479353650478908, 1: 0.5024482701784585,..."
4,0.948276,0.954772,0.866667,0.970149,0.915493,0.939394,0.881297,155,10,2,65,3,2,recording_20260225_143640.csv,232,501,"{0: 0.4479353650478908, 1: 0.5024482701784585,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,0.855799,0.750000,1.000000,0.500000,0.666667,1.000000,0.644787,227,0,46,46,12,10,recording_20260225_143209.csv,319,501,"{0: 0.39292963549876575, 1: 0.7585105406869779..."
66,0.903974,0.832990,0.982759,0.670588,0.797203,0.995392,0.760312,216,1,28,57,12,10,recording_20260225_143246.csv,302,501,"{0: 0.39292963549876575, 1: 0.7585105406869779..."
67,0.943966,0.902985,1.000000,0.805970,0.892562,1.000000,0.864354,165,0,13,54,12,10,recording_20260225_143640.csv,232,501,"{0: 0.39292963549876575, 1: 0.7585105406869779..."
68,0.925490,0.866197,1.000000,0.732394,0.845528,1.000000,0.814767,184,0,19,52,12,10,recording_20260225_143708.csv,255,501,"{0: 0.39292963549876575, 1: 0.7585105406869779..."


In [7]:
# Optional: save result tables
output_dir = ROOT_DIR / "data/05_results/hmm"
output_dir.mkdir(parents=True, exist_ok=True)

model_summary_path = output_dir / "hmm_validation_model_summary.csv"
per_file_path = output_dir / "hmm_validation_per_file.csv"

model_summary.to_csv(model_summary_path, index=False)
per_file_results.to_csv(per_file_path, index=False)

print(f"Saved: {model_summary_path}")
print(f"Saved: {per_file_path}")

Saved: /Users/jole/Studium/HandPose/HandPoseEstimation/data/05_results/hmm/hmm_validation_model_summary.csv
Saved: /Users/jole/Studium/HandPose/HandPoseEstimation/data/05_results/hmm/hmm_validation_per_file.csv
